<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/torch_ntuple_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:
- Build plots for BitBullyArena Results!

In [ ]:
!ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N "" -q
!ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts
!cat ~/.ssh/id_rsa.pub

Add key here:
https://github.com/settings/keys

In [ ]:
!ssh -T git@github.com
!git config --global user.email "8896197+MarkusThill@users.noreply.github.com"
!git config --global user.name "Markus Thill"
!git clone git@github.com:MarkusThill/techdays26.git
!pip install -e techdays26[dev,lab1]

In [1]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [ ]:
from techdays26.ntuples import NTUPLE_BITIDX_LIST

In [ ]:
import torch
import torch.nn as nn

class TDConnect4AgentTorch:
    """TD n-tuple agent compatible with `Connect4Agent` Protocol.

    Implements:
      - score_all_moves(board) -> dict[int,int]
      - best_move(board) -> int
      - score_move(board, column, first_guess=0) -> int
    """

    def __init__(
        self,
        model_path: str | None=None,
        *,
        tie_break: str = "center",
    ) -> None:
        net2 = NTupleNetwork.load(model_path, device="cpu")
        net2.eval()
        self._tie_break = tie_break
        self._eval = net2

    # ---- Protocol method 1 ----
    def score_all_moves(self, board) -> dict[int, int]:
        """Return {col: score} for all legal moves. Illegal/full columns are excluded."""
        player_to_move = board.current_player() - 1  # BitBully: {1,2} -> {0,1}
        if player_to_move not in (0, 1):
            raise ValueError(f"Unexpected current_player(): {board.current_player()}")

        scores: dict[int, int] = {}

        for col in range(7):
            if not board.is_legal_move(col):
                continue

            score = self.score_move(board=board, column=col)
            scores[col] = score

        return scores

    # ---- Protocol method 2 ----
    def best_move(self, board) -> int:
        """Return best legal move using BitBully-like tie breaking."""
        scores = self.score_all_moves(board)
        if not scores:
            raise ValueError("No legal moves available.")

        best = max(scores.values())
        candidates = [c for c, v in scores.items() if v == best]

        if len(candidates) == 1:
            return candidates[0]

        if self._tie_break == "center":
            center = 3
            return min(candidates, key=lambda c: (abs(c - center), c))
        if self._tie_break == "left":
            return min(candidates)
        if self._tie_break == "right":
            return max(candidates)

        raise ValueError("tie_break must be one of: 'center', 'left', 'right'")

    # ---- Optional Protocol method ----
    def score_move(self, board, column: int, first_guess: int = 0) -> int:
        """Evaluate a single legal move. `first_guess` is accepted for Protocol compatibility."""
        _ = first_guess  # this TD agent ignores it
        if not board.is_legal_move(column):
            raise ValueError(f"Illegal move: column {column}")

        player_to_move = board.current_player()
        after = board.play_on_copy(column)

        if after.has_win():
            return 100

        all_tokens, active_tokens, moves_left = after._board.rawState()
        torch_board = BoardBatch(
            all_tokens=torch.tensor([all_tokens]),
            active_tokens=torch.tensor([active_tokens]),
            moves_left=torch.tensor([moves_left]),
        )

        score = float(self._eval.forward(torch_board)[0].item())

        if player_to_move == 2:
            score = -score

        return int(score * 100.0)

class NTupleNetwork(nn.Module):
    def __init__(self, n_tuple_list: list[list[int]]):
        super().__init__()

        self.M = len(n_tuple_list)
        self.N = len(n_tuple_list[0])
        self.K = 4**self.N

        # [M, N] bit indices
        self.n_tuple_tensor = torch.tensor(n_tuple_list, dtype=torch.int64)

        # Two players × M LUTs × K entries
        # 0 = Yellow, 1 = Red
        self.W = nn.Parameter(torch.zeros(2, self.M, self.K))

    def forward(self, board: "BoardBatch") -> torch.Tensor:
        """
        Returns:
            [B] tensor in [-1, 1]
        """
        # [B, M] table indices
        T = board.table_positions(self.n_tuple_tensor)
        T_mir = board.mirror().table_positions(self.n_tuple_tensor)
        B, M = T.shape
        dev = T.device

        # Active player per board: 0=Yellow, 1=Red
        # reuse your parity logic
        player_idx = ((board.moves_left.to(torch.int64) & 1) != 0).to(torch.int64)
        # shape: [B]

        # Pattern indices [M]
        m_idx = torch.arange(M, device=dev)

        # Gather:
        # W[player_idx[b], m, T[b,m]]
        w = self.W[
            player_idx.unsqueeze(1),  # [B,1]
            m_idx.unsqueeze(0),  # [1,M]
            T,  # [B,M]
        ]  # -> [B,M]
        w_mir = self.W[player_idx.unsqueeze(1), m_idx.unsqueeze(0), T_mir]

        # Sum over patterns (and symmetry): [B]
        score = (w + w_mir).sum(dim=1)
        return torch.tanh(score)

    @torch.no_grad()
    def save(self, path: str) -> None:
        """Save model weights + architecture-relevant metadata."""
        payload = {
            "state_dict": self.state_dict(),
            "n_tuple_list": self.n_tuple_tensor.cpu().tolist(),
        }
        torch.save(payload, path)

    @classmethod
    def load(
        cls,
        path: str,
        *,
        device: torch.device | str = "cpu",
        strict: bool = True,
    ) -> "NTupleNetwork":
        """
        Load model fully onto the specified device (CPU or GPU).
        """

        # 1) Always load checkpoint onto CPU first (safe & portable)
        payload = torch.load(path, map_location="cpu")

        if not isinstance(payload, dict) or "state_dict" not in payload:
            raise ValueError("Invalid checkpoint format.")

        n_tuple_list = payload.get("n_tuple_list")
        if n_tuple_list is None:
            raise ValueError("Checkpoint missing 'n_tuple_list'.")

        # 2) Construct model (still on CPU)
        model = cls(n_tuple_list=n_tuple_list)

        # 3) Load weights (still CPU tensors)
        model.load_state_dict(payload["state_dict"], strict=strict)

        # 4) Move EVERYTHING (parameters + buffers) in one go
        model = model.to(device)

        return model

import torch
import torch.nn.functional as F


def best_afterstate_values(
    board: "BoardBatch",
    net: "NTupleNetwork",
    *,
    moves_mask: torch.Tensor | None = None,
    randomize: torch.Tensor | None = None,  # [B] bool (epsilon-greedy)
    use_non_losing: bool = True,
    no_grad: bool = True,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Returns:
        chosen_mv:  [B] int64 one-hot move mask (landing square)
        chosen_val: [B] float32 value (reward if terminal else net score)
    """
    dev = board.all_tokens.device
    B = board.all_tokens.shape[0]

    # Which move set to iterate?
    if moves_mask is None:
        moves_mask = (
            board.generate_non_losing_moves()
            if use_non_losing
            else board.generate_legal_moves()
        )
    moves_mask = moves_mask.to(device=dev, dtype=torch.int64)

    yellow_to_move = (board.moves_left.to(torch.int64) & 1) == 0  # [B] bool

    neg_inf = torch.tensor(float("-inf"), device=dev)
    pos_inf = torch.tensor(float("inf"), device=dev)
    best_val = (
        torch
        .where(yellow_to_move, neg_inf, pos_inf)
        .to(torch.float32)
        .expand(B)
        .clone()
    )
    best_mv = torch.zeros((B,), device=dev, dtype=torch.int64)

    # Uniform random move via reservoir sampling over the iterated moves
    rand_mv = torch.zeros((B,), device=dev, dtype=torch.int64)
    rand_val = torch.full((B,), float("nan"), device=dev, dtype=torch.float32)
    seen = torch.zeros(
        (B,), device=dev, dtype=torch.int32
    )  # number of moves seen so far per board

    ctx = torch.no_grad() if no_grad else torch.enable_grad()
    with ctx:
        for mv in board.iter_move_masks(moves_mask):
            active = mv != 0
            if not active.any():
                break

            # Afterstate
            tmp = type(board)(
                all_tokens=board.all_tokens.clone(),
                active_tokens=board.active_tokens.clone(),
                moves_left=board.moves_left.clone(),
            )
            legal = tmp.play_masks(mv)
            active = active & legal  # defensive

            # Terminal reward: +1/-1/0 or NaN if not done
            r = tmp.reward().to(torch.float32)
            v = net(tmp).to(torch.float32)

            # tiebreak noise (optional)
            eps = 1e-4
            v = v + eps * torch.randn_like(v)

            val = torch.where(torch.isnan(r), v, r)  # [B]

            # --- greedy best update ---
            better = (
                torch.where(yellow_to_move, val > best_val, val < best_val) & active
            )
            best_val = torch.where(better, val, best_val)
            best_mv = torch.where(better, mv, best_mv)

            # --- reservoir sampling (uniform random among legal moves) ---
            # increment seen count for boards where this move exists
            seen = seen + active.to(seen.dtype)  # t in {1..}
            # replace current random choice with prob 1/seen
            # (uniform float u in [0,1); replace if u < 1/t)
            t = seen.to(torch.float32)
            replace = active & (torch.rand((B,), device=dev) < (1.0 / t))
            rand_mv = torch.where(replace, mv, rand_mv)
            rand_val = torch.where(replace, val, rand_val)

    if randomize is None:
        return best_mv, best_val

    randomize = randomize.to(device=dev, dtype=torch.bool)
    chosen_mv = torch.where(randomize, rand_mv, best_mv)
    chosen_val = torch.where(randomize, rand_val, best_val)
    return chosen_mv, chosen_val

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent
import bitbully.bitbully_core as bbc
from bitbully import Board

bitbully_agent = BitBully(opening_book="12-ply-dist", tie_break="random")

In [ ]:
import logging
from techdays26.bitbully_arena import (
    BitBullyArena,
    ArenaConfig,
    AgentSpec,
    RandomAgent,
    Color,
    TimeControl,
    ArenaResult,
    GameRecord,
    AggregateRow,
    TerminationReason,
    SkippedPairing,
    MoveMeta,
    GamePlayers,
    GameConfig,
    get_table_legend,
    format_aggregate_table
)

def evaluate(td_agent, rnd_agent, bitbully_agent):
  # Logger is optional; arena is silent by default unless you configure logging.
  logger = logging.getLogger("bitbully.arena")
  logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout


  cfg = ArenaConfig(
      agents=(
          AgentSpec(
              agent_id="random",
              agent=rnd_agent,
              colors=(Color.YELLOW, Color.RED),  # can play both
              epsilons=(0.00,),
          ),
          AgentSpec(
              agent_id="bitbully",
              agent=bitbully_agent,
              colors=(Color.YELLOW, Color.RED),  # can play both
              epsilons=(0.00, 0.1, 0.2, 0.3),
          ),
          AgentSpec(
              agent_id="ntuple",
              agent=td_agent,
              colors=(Color.YELLOW, Color.RED),  # can play both,
              epsilons=(0.00,),
          ),
      ),
      n_games=50,  # n games per pairing per ε per color-swap
      time_control=TimeControl(
          per_move_timeout_s=4.0,  # best-effort (measured after return)
          per_game_budget_s=45.0,  # seconds of total think time
      ),
      seed=None,
      log_scores=False,  # set True to also call score_all_moves() for logging
      use_tqdm=True,  # optional progress bar
      logger=logger,
  )

  # -----------------------------
  # Run arena
  # -----------------------------

  arena = BitBullyArena()
  result = arena.run(cfg)
  return result



In [1]:
from techdays26 import utils
from techdays26.torch_board import BoardBatch
commit_sha = utils.get_commit_hash(".")
reqs = utils.get_requirements_string()

In [ ]:
from pathlib import Path


pre_trained_model: str | None = None
train_folder: str | Path = "/content/drive/MyDrive/models/exp_20260207/"
n_evaluate = 1000

train_folder = Path(train_folder)
train_folder.mkdir(parents=True, exist_ok=True)

B = 30000
epsilon = 0.1

# load pre-trained, if desired:
if pre_trained_model:
    ntuple_net = NTupleNetwork.load(
        pre_trained_model,
        device=device,
    )
    # ntuple_net.eval() # DO NOT DO DURING TRAINING
else:
    ntuple_net = NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST).to(device)

torch_board = BoardBatch.empty(B, device)
bb_board = [bbc.BoardCore() for _ in range(B)]

assert ntuple_net.training, "Model should be in training mode!"

# opt = torch.optim.Adam(
#    ntuple_net.parameters(),
#    lr=1e-3,        # start here
#    betas=(0.9, 0.999),
#    eps=1e-8,
# )

# opt = torch.optim.AdamW(
#    ntuple_net.parameters(),
#    lr=1e-5,
#    weight_decay=0.01,  # optional, small
# )

opt = torch.optim.Adam(ntuple_net.parameters(), lr=1e-2) # used to be 1e-3
scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.9999)

all_results = []
losses = []
for step in range(100000):
    V_old = ntuple_net(torch_board)
    randomize = (
        torch.rand((B,), device=torch_board.all_tokens.device) < epsilon
    )  # [B] bool

    with torch.no_grad():
        best_mv, V_new = best_afterstate_values(
            torch_board,
            ntuple_net,
            moves_mask=None,
            randomize=randomize,
            use_non_losing=False,
            no_grad=True,
        )

    # At any time, any move has to be legal:
    legal = torch_board.play_masks(best_mv)
    if False:
        assert all(legal == True) and not any(legal == False)

    # If the games are over, there has to be a reward
    if False:
        assert (torch_board.done() & torch.isnan(torch_board.reward())).sum() == 0
    done = torch_board.done()

    # Update only for afterstates which are terminal states or not random moves
    update_mask = done | (~randomize)

    loss = torch.nn.functional.mse_loss(V_old[update_mask], V_new[update_mask])

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    scheduler.step()

    losses.append(loss.item())
    if step % 100 == 0:
        lr = opt.param_groups[0]["lr"]
        print(f"step {step:6d} | lr = {lr:.3e} | loss = {loss.item():.6f}")

    if step % n_evaluate == 0:
        print("evaluate...")
        # Save the weights to a file
        weights_path = f"{train_folder}/step_{step}_model_weights_loss_{loss.item():.3f}.pt"
        ntuple_net.save(weights_path)
        td_agent = TDConnect4AgentTorch( # TODO: Allow to pass object (or copy) directly
            model_path=weights_path
        )
        result = evaluate(td_agent, RandomAgent(), bitbully_agent)
        save_arena_result(result, f"{train_folder}/step_{step}_arena_result.json") # TODO: Integrate into class

        out_file = Path(train_folder / "0_log.txt")

        with out_file.open("a", encoding="utf-8") as f:
            if step == 0:
                f.write(f"Git commit SHA: {commit_sha}\n")
                f.write(f"Requirements: {reqs}\n")
                f.write(get_table_legend() + "\n\n")
            lr = opt.param_groups[0]["lr"]
            f.write("=====================================================" * 2 + "\n")
            f.write(f"step {step:6d} | lr = {lr:.3e} | loss = {loss.item():.5f}\n\n")
            f.write(format_aggregate_table(result))
            f.write("=====================================================" * 2 + "\n\n")


    # Just some verification
    if False:
        for b_idx in range(B):
            mv = best_mv[b_idx].item()
            a = move_mask_to_column(mv)
            assert bb_board[b_idx].play(a)
            bb_done = bb_board[b_idx].hasWin() or bb_board[b_idx].movesLeft() <= 0

            assert bb_done == done[b_idx].item(), f"Problem with {b_idx}"
            if bb_done:
                if bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() % 2 == 0:
                    # print(best_val[b_idx].item(), bb_board[b_idx])
                    assert V_new[b_idx].item() == -1
                elif bb_board[b_idx].hasWin() and bb_board[b_idx].movesLeft() % 2 == 1:
                    assert V_new[b_idx].item() == 1
                    # print(best_val[b_idx].item(), bb_board[b_idx])
                elif bb_board[b_idx].movesLeft() <= 0:
                    assert V_new[b_idx].item() == 0
                bb_board[b_idx].setRawState(0, 0, 42)

    if done.any():
        torch_board.reset(done)

step      0 | lr = 9.999e-03 | loss = 0.000000
evaluate...


BitBullyArena: 100%|██████████| 900/900 [00:37<00:00, 24.06game/s]


step    100 | lr = 9.900e-03 | loss = 0.191849
step    200 | lr = 9.801e-03 | loss = 0.087358
step    300 | lr = 9.703e-03 | loss = 0.109112
step    400 | lr = 9.607e-03 | loss = 0.133101
step    500 | lr = 9.511e-03 | loss = 0.225295
step    600 | lr = 9.417e-03 | loss = 0.095951
step    700 | lr = 9.323e-03 | loss = 0.118558
step    800 | lr = 9.230e-03 | loss = 0.242427
step    900 | lr = 9.138e-03 | loss = 0.141312
step   1000 | lr = 9.047e-03 | loss = 0.226287
evaluate...


BitBullyArena: 100%|██████████| 900/900 [00:53<00:00, 16.68game/s]


step   1100 | lr = 8.957e-03 | loss = 0.204385
step   1200 | lr = 8.868e-03 | loss = 0.287604
step   1300 | lr = 8.780e-03 | loss = 0.304410
step   1400 | lr = 8.693e-03 | loss = 0.150778
step   1500 | lr = 8.606e-03 | loss = 0.099212
step   1600 | lr = 8.521e-03 | loss = 0.086219
step   1700 | lr = 8.436e-03 | loss = 0.089833
step   1800 | lr = 8.352e-03 | loss = 0.085637
step   1900 | lr = 8.269e-03 | loss = 0.104211
step   2000 | lr = 8.186e-03 | loss = 0.096712
evaluate...


...

step  28100 | lr = 6.019e-04 | loss = 0.033099
step  28200 | lr = 5.959e-04 | loss = 0.034187
step  28300 | lr = 5.900e-04 | loss = 0.033100
step  28400 | lr = 5.841e-04 | loss = 0.031669
step  28500 | lr = 5.783e-04 | loss = 0.032045
step  28600 | lr = 5.725e-04 | loss = 0.036276
step  28700 | lr = 5.669e-04 | loss = 0.033799
step  28800 | lr = 5.612e-04 | loss = 0.036096
step  28900 | lr = 5.556e-04 | loss = 0.038173
step  29000 | lr = 5.501e-04 | loss = 0.035788
evaluate...


BitBullyArena: 100%|██████████| 900/900 [01:23<00:00, 10.75game/s]


step  29100 | lr = 5.446e-04 | loss = 0.032967
step  29200 | lr = 5.392e-04 | loss = 0.033662
step  29300 | lr = 5.338e-04 | loss = 0.027468
step  29400 | lr = 5.285e-04 | loss = 0.029041
step  29500 | lr = 5.233e-04 | loss = 0.029270
step  29600 | lr = 5.181e-04 | loss = 0.031820
step  29700 | lr = 5.129e-04 | loss = 0.035585
step  29800 | lr = 5.078e-04 | loss = 0.038640
step  29900 | lr = 5.027e-04 | loss = 0.031801
step  30000 | lr = 4.977e-04 | loss = 0.031770
evaluate...


BitBullyArena: 100%|██████████| 900/900 [01:27<00:00, 10.32game/s]


step  30100 | lr = 4.928e-04 | loss = 0.027920
step  30200 | lr = 4.879e-04 | loss = 0.028716
step  30300 | lr = 4.830e-04 | loss = 0.029932
step  30400 | lr = 4.782e-04 | loss = 0.027031
step  30500 | lr = 4.735e-04 | loss = 0.028775
step  30600 | lr = 4.688e-04 | loss = 0.025841
step  30700 | lr = 4.641e-04 | loss = 0.025712


In [ ]:
result = load_arena_result(f"{train_folder}/arena_result.json")

In [ ]:
if False: # Old:
  print("==================================================="*4)
  print("Skipped pairings:", len(result.skipped))
  print("Games played:", len(result.games))
  print("Aggregate rows:", len(result.aggregates))

  # Print aggregate rows (one row per (yellow_agent, red_agent, eps_y, eps_r))
  for row in result.aggregates:
      print(row)

  # Look at a single game record (includes move list + per-move metadata)
  g0 = result.games[0]
  print("First game:", g0.game_cfg)
  print("Termination:", g0.termination, "winner:", g0.winner)
  print("Moves:", g0.moves)
  print("First move meta:", g0.move_meta[0])
  print("==================================================="*4, "\n")

In [ ]:
agents: dict[str, Connect4Agent] = {
    "BitBully-12ply": bitbully_agent,
    "TD-Agent": td_agent,
}

In [ ]:
from bitbully.gui_c4 import GuiC4

%matplotlib ipympl

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())